```sql
CREATE TABLE orders (
    id BIGINT GENERATED ALWAYS AS IDENTITY
        PRIMARY KEY,

    user_id BIGINT NOT NULL,

    amount NUMERIC(12, 2) NOT NULL
        CHECK (amount > 0),

    status TEXT NOT NULL
        CHECK (
            status IN (
                'pending',
                'paid',
                'cancelled'
            )
        ),

    created_at TIMESTAMPTZ NOT NULL
        DEFAULT CURRENT_TIMESTAMP
);
```

#### SELECT 查询列
##### 查询全部
`SELECT * FROM orders;`
实际生产不要用*，因为性能差，而是选择性的某些字段。

##### 字段别名
`SELECT id AS order_id, amount AS total_amount FROM orders;`
AS可以省略，但是可读性更好

#### WHERE 过滤行
``` sql
SELECT
    id,
    user_id,
    amount
FROM orders
WHERE status = 'paid';
```
多个条件可以用AND 或 OR  或 IN 连接。
1. `WHERE status = 'paid' OR status = 'pending';`
2. `WHERE status IN ('paid', 'pending');`
3. `WHERE status = 'paid' AND amount >= 200;`

不等于使用`<>`

##### 范围——between
`WHERE amount BETWEEN 100 AND 300`

##### 集合——in
`WHERE status IN ('paid', 'pending')`

##### 模糊查询
`WHERE username LIKE 'Ali%'`
> ILIKE可以进行不区分大小写的匹配

##### NULL
要用is
`WHERE deleted_at IS NULL`

### AND 和 OR 的优先级
```sql
WHERE status = 'paid'
   OR status = 'pending'
  AND amount >= 100
```
AND 的优先级高于 OR。

```sql
WHERE (
        status = 'paid'
        OR status = 'pending'
      )
  AND amount >= 100;
```
可以加括号来改变优先级

### ORDER BY
```sql
SELECT *
FROM orders
ORDER BY amount ASC;
```
按照金额从低到高
- ASC 从低到高
- DESC 从高到低

可以按照多个字段排序
```sql
ORDER BY amount ASC, created_at DESC;
```
1. 先按创建时间降序
2. 创建时间相同时，再按 ID 降序

### LIMIT 与 OFFSET
```sql
SELECT *
FROM orders
ORDER BY id
LIMIT 10
OFFSET 10;
```
- OFFSET跳过前10条
- LIMIT取10条

> 为什么需要order by？
> 数据库没有义务每次使用相同顺序，所以如果没有orderby可能会出现重复数据、遗漏数据、顺序变化。

### 聚合函数
聚合函数把多行数据计算成一个结果。
- COUNT()
- SUM()
- AVG()
- MIN()
- MAX()

订单数量:
```sql
SELECT COUNT(*)
FROM orders;
```

订单总金额:
```sql
SELECT SUM(amount)
FROM orders;
```

订单平均金额:
```sql
SELECT AVG(amount)
FROM orders;
```

#### COUNT(*) 与 COUNT(column)
COUNT(*): 统计行数，不关心某一列是否为 NULL。
COUNT(column): 统计某一列非 NULL 的行数。

> COALESCE 返回第一个非 NULL 值. COALESCE(NULL, 0)  -- 0

### GROUP BY

In [ ]:
SELECT
    user_id,
    COUNT(*) AS order_count
FROM orders
GROUP BY user_id;

按 user_id 分组

→ user_id=10 的订单放在一组

→ user_id=20 的订单放在一组

→ 每组分别 COUNT

### WHERE 与 HAVING
`WHERE`过滤原始行，在分组和聚合前执行
`HAVING`过滤分组后的组，在分组和聚合后执行

### 子查询
子查询是嵌套在查询中的查询。

### 书写顺序与逻辑处理顺序
```sql
SELECT
    user_id,
    SUM(amount) AS total
FROM orders
WHERE status = 'paid'
GROUP BY user_id
HAVING SUM(amount) >= 250
ORDER BY total DESC
LIMIT 10;
```
```
执行顺序：
1. FROM
2. WHERE
3. GROUP BY
4. HAVING
5. SELECT
6. ORDER BY
7. LIMIT / OFFSET
```

### 子查询
子查询就是嵌套在另一条 SQL 中的查询。
#### 标量子查询
标量子查询必须返回一个值，因为一个比较位置只需要一个值。
```sql
SELECT *
FROM orders
WHERE amount > (
    SELECT AVG(amount)
    FROM orders
);
```

#### IN 子查询
```sql
SELECT *
FROM users
WHERE id IN (
    SELECT user_id
    FROM orders
    WHERE status = 'paid'
);
```
users.id 是否位于这个结果集合中.

#### EXISTS 子查询
```sql
SELECT *
FROM users AS u
WHERE EXISTS (
    SELECT 1
    FROM orders AS o
    WHERE o.user_id = u.id
      AND o.status = 'paid'
);
```
EXISTS 判断的是：子查询是否至少返回一行。

引用了外部查询当前用户，这叫相关子查询。
`SELECT 1` 并不是在返回业务字段；对 `EXISTS` 来说，重点只是有没有匹配行。

### 子查询是不是一定比 JOIN 慢？
PostgreSQL 优化器可能把不同 SQL 形式转换为相似的执行计划。

性能取决于：
- 数据量
- 索引
- 过滤条件
- 数据分布
- 执行计划
- 查询写法
- PostgreSQL 版本和统计信息

两者表达的语义可能不同，优化器也可能重写查询。实际性能需要结合 EXPLAIN 或 EXPLAIN ANALYZE、索引和数据分布判断。

### 总结：

SQL 查询在逻辑上先由 FROM 确定数据来源，再由 WHERE 过滤原始行，然后通过 GROUP BY 分组、HAVING 过滤组，最后计算 SELECT、排序并分页。WHERE 不能直接过滤聚合结果，因为它发生在聚合之前；聚合条件应该使用 HAVING。